# TelecomTS: understand the prepared data

Follow one window from its CSV row to its KPI values, then explore the whole dataset.

- **Scenario/window:** one short recording with several KPI channels measured together.
- **KPI:** one measurement, such as received bytes or an error rate.
- **Labels:** the dataset's answers about fault presence and fault type, shown here for inspection.

Run `uv sync --extra dev --extra analysis` and prepare data with the ingestion CLI first. This notebook reads `data/scenarios/` and saves figures in `artifacts/telecomts_eda/`. Override those locations with `TELECOMTS_SCENARIO_DIR` and `TELECOMTS_EDA_DIR`.

Train/test splits and reviewed text are still needed before experiments (issue #22).

In [ ]:
import json
import os
import sys
from pathlib import Path

%matplotlib inline
import matplotlib.pyplot as plt
import pandas as pd
from IPython.display import Markdown, display

ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "pyproject.toml").exists() and (p / "src/clauseguard").is_dir())
sys.path.insert(0, str(ROOT / "src"))
from clauseguard.datasets.telecomts import file_sha256

DATA = Path(os.environ.get("TELECOMTS_SCENARIO_DIR", ROOT / "data/scenarios"))
FIGURES = Path(os.environ.get("TELECOMTS_EDA_DIR", ROOT / "artifacts/telecomts_eda"))
FIGURES.mkdir(parents=True, exist_ok=True)
if not (DATA / "csv_schema.json").exists():
    raise FileNotFoundError(f"Prepare analysis CSVs first; missing {DATA / 'csv_schema.json'}")
schema = json.loads((DATA / "csv_schema.json").read_text())
for name, expected_hash in schema["files"].items():
    assert file_sha256(DATA / name) == expected_hash, f"Changed CSV: {name}"
assert file_sha256(DATA / "scenarios.jsonl") == schema["source_scenarios_sha256"], "CSV/source mismatch"
overview = pd.read_csv(DATA / "scenarios_overview.csv", dtype=str, keep_default_na=False)
assert len(overview) == schema["scenarios"]
if overview.empty:
    raise ValueError("No accepted scenarios available; inspect rejected.jsonl and conversion.json")
conversion = json.loads((DATA / "conversion.json").read_text()) if (DATA / "conversion.json").exists() else {}

def show_plot(name):
    plt.tight_layout()
    plt.savefig(FIGURES / f"{name}.png", dpi=130, bbox_inches="tight")
    plt.show()

display(pd.DataFrame([conversion.get("counts", {"accepted": len(overview)})]))
print("Source hash:", schema["source_scenarios_sha256"])
print("Splits are not created or verified by this notebook.")

## First, look at the CSV files

- `scenarios_overview.csv`: one row per window, with labels and source details.
- `kpi_samples.csv`: one row per time step, with values for every KPI.
- `scenario_id` joins the two files. `sample_index` starts at 0 inside each window.
- A `__missing` column marks missing values. The category string `None` is kept as text.

These are the first five rows of each file. The sample preview reads only five rows from the large CSV.

In [ ]:
print("scenarios_overview.csv — first 5 rows")
display(overview.head())
print("kpi_samples.csv — first 5 rows")
samples_head = pd.read_csv(DATA / "kpi_samples.csv", nrows=5, dtype=str, keep_default_na=False)
display(samples_head)

## What do the labels mean?

There are three separate questions:

- **`origin`:** where did this window come from? `normal` = no labelled fault; `real` = controlled jamming in the testbed; `synthetic` = an injected fault; `unknown` = origin not established.
- **`fault_label`:** is a fault marked? `normal` = no; `abnormal` = yes.
- **`rca_label`:** which fault type is marked? The table below explains the source names. `unknown` means the cause is unsupported or missing.

The chart shows origins on the left and abnormal fault types on the right. Normal windows are counted separately so they do not hide the much smaller fault categories. These labels describe dataset annotations; they do not independently prove a cause.

In [ ]:
label_meanings = {
    "normal": "No fault is marked in the window.",
    "Jamming": "Controlled radio interference from a jammer.",
    "Antenna Failure": "A simulated antenna fault.",
    "Buffer Overflow (Gradual Buildup)": "A buffer gradually fills beyond its capacity.",
    "Co-Channel Interference (Mild)": "Mild interference on the same radio channel.",
    "Co-Channel Interference (Severe)": "Severe interference on the same radio channel.",
    "Doppler Shift (Severe)": "A strong simulated radio-frequency shift associated with movement.",
    "Faulty Handover Algorithm (Too Frequent)": "Too many handovers between connections/cells.",
    "Faulty RF Filters (Temporal)": "A simulated radio-filter fault that changes over time.",
    "High Network Congestion (Gradual Buildup)": "Network load and congestion rise gradually.",
    "High Network Congestion (Sudden Spike)": "Network congestion rises suddenly.",
    "Resource Allocation Bugs": "A simulated fault in assigning network resources.",
    "unknown": "A fault is marked, but its cause is unsupported or missing.",
}
label_counts = overview["rca_label"].value_counts()
label_table = pd.DataFrame({"Label": list(label_meanings), "Meaning": list(label_meanings.values())})
label_table["Windows"] = label_table["Label"].map(label_counts).fillna(0).astype(int)
with pd.option_context("display.max_colwidth", None):
    display(label_table)
display(overview["fault_label"].value_counts().rename_axis("Fault label").to_frame("Windows"))

fig, axes = plt.subplots(1, 2, figsize=(16, 6), gridspec_kw={"width_ratios": [1, 2]})
origin_names = {"normal": "Normal", "real": "Controlled jamming", "synthetic": "Injected fault", "unknown": "Unknown"}
origin_counts = overview["origin"].value_counts().sort_values()
origin_counts.rename(index=origin_names).plot.barh(ax=axes[0], color="#4c78a8")
for container in axes[0].containers:
    axes[0].bar_label(container, fmt="%.0f", padding=4)
axes[0].set_title("Where the windows came from")
axes[0].set_xlabel("Windows")
axes[0].set_ylabel("")
axes[0].set_xlim(0, origin_counts.max() * 1.25)
fault_counts = overview.loc[overview["fault_label"].eq("abnormal"), "rca_label"].value_counts().sort_values()
if not fault_counts.empty:
    fault_counts.plot.barh(ax=axes[1], color="#e45756")
    for container in axes[1].containers:
        axes[1].bar_label(container, fmt="%.0f", padding=4)
    axes[1].set_xlim(0, fault_counts.max() * 1.2)
else:
    axes[1].text(0.5, 0.5, "No abnormal windows in this selection", ha="center", transform=axes[1].transAxes)
axes[1].set_title("Fault types (abnormal windows only)")
axes[1].set_xlabel("Windows")
axes[1].set_ylabel("")
show_plot("origin_and_categories")

## What is inside a window?

- **Length:** the number of time steps in one window (`sample_count`).
- **Sampling rate/cadence:** measurements per second. At 10 Hz, samples are 0.1 seconds apart.
- **Time span:** the first-to-last sample gap is `(sample_count - 1) / sampling_rate_hz`. For 128 samples at 10 Hz, that is 12.7 seconds.
- **Channel:** one KPI measured throughout the window. Numeric channels hold numbers; categorical channels hold names such as `TCP` or `UDP`.
- **Units:** `unknown` means the preparation run did not supply a verified unit. Source timestamps keep their original timezone uncertainty.

The table shows the shapes actually present in this export; the diagram illustrates its first window.

In [ ]:
window_shapes = overview.groupby(["sample_count", "sampling_rate_hz"], sort=True).size().reset_index(name="windows")
display(window_shapes)
channel_info = pd.DataFrame.from_dict(schema["channels"], orient="index").rename_axis("KPI")
channel_info["unit"] = channel_info["unit"].fillna("unknown")
display(channel_info)

first_window = overview.iloc[0]
length = int(first_window["sample_count"])
rate = float(first_window["sampling_rate_hz"])
times = [index / rate for index in range(length)]
fig, axes = plt.subplots(1, 2, figsize=(12, 3.5))
axes[0].plot(times, [0] * length, "o-", markersize=3)
axes[0].set_yticks([])
axes[0].set_xlabel("Seconds since this window started")
axes[0].set_title(f"One window: {length} samples at {rate:g} Hz")
axes[0].text(0.5, 0.8, f"Sample gap: {1 / rate:g} s | First-to-last span: {times[-1]:g} s", ha="center", transform=axes[0].transAxes)
channel_info["kind"].value_counts().plot.barh(ax=axes[1], color="#72b7b2")
for container in axes[1].containers:
    axes[1].bar_label(container, padding=4)
axes[1].set_title("KPI channel types in this export")
axes[1].set_xlabel("Channels")
axes[1].set_ylabel("")
axes[1].margins(x=0.2)
show_plot("lengths_and_cadence")

## Missing values across the entire selected export

Scan CSV chunks to bound memory. Missing masks preserve the distinction between a real null and categorical strings such as `None` or an empty string. The distribution and correlation plots below use a small, deterministic illustrative selection, not the whole dataset.

In [ ]:
names = sorted(schema["channels"])
mask_columns = [name + "__missing" for name in names]
selected_ids = set()
for _, group in overview.groupby(["origin", "rca_label"], sort=True):
    selected_ids.update(group.sample(n=min(3, len(group)), random_state=42)["scenario_id"])
missing = pd.Series(0, index=mask_columns, dtype="int64")
observed_samples = 0
pieces = []
columns = ["scenario_id", "sample_index", "relative_time_seconds", *names, *mask_columns]
for chunk in pd.read_csv(DATA / "kpi_samples.csv", usecols=columns, dtype=str, keep_default_na=False, chunksize=50000):
    observed_samples += len(chunk)
    missing += chunk[mask_columns].eq("True").sum()
    selected = chunk.loc[chunk["scenario_id"].isin(selected_ids)]
    if not selected.empty:
        pieces.append(selected)
assert observed_samples == schema["samples"]
missing.index = names
(missing / observed_samples * 100).sort_values().plot.barh(figsize=(9, 5), title="Missing values across all exported samples (%)")
show_plot("missing_values")
sample = pd.concat(pieces, ignore_index=True)
sample["sample_index"] = pd.to_numeric(sample["sample_index"])
sample["relative_time_seconds"] = pd.to_numeric(sample["relative_time_seconds"])
numeric = [name for name in names if schema["channels"][name]["kind"] == "numeric"]
for name in numeric:
    sample[name] = pd.to_numeric(sample[name].where(sample[name + "__missing"].eq("False")), errors="raise")
print(f"Illustrative selection: {len(selected_ids)} windows, {len(sample)} samples; full export: {observed_samples} samples.")

## Read one window: KPIs, time steps, and values

We choose an abnormal synthetic window when one is available; otherwise, we use another available window. The table lists each KPI and its first eight values. The next table shows the first ten time steps, so you can see how channels line up.

The shaded area in the figure is the source's labelled fault interval (inclusive sample indices). It is an annotation, not a model prediction. A window can have a fault label without known start/end indices.

In [ ]:
candidates = overview.loc[overview["scenario_id"].isin(selected_ids)]
synthetic = candidates.loc[candidates["origin"].eq("synthetic") & candidates["fault_label"].eq("abnormal")]
abnormal = candidates.loc[candidates["fault_label"].eq("abnormal")]
if not synthetic.empty:
    example = synthetic.iloc[0]
elif not abnormal.empty:
    example = abnormal.iloc[0]
else:
    example = candidates.iloc[0]
example_id = example["scenario_id"]
with (DATA / "scenarios.jsonl").open() as source:
    for line in source:
        example_record = json.loads(line)
        if example_record["scenario_id"] == example_id:
            break
    else:
        raise ValueError(f"Window missing from source JSONL: {example_id}")
window = example_record["kpi_window"]
print(f"Window: {example_id}")
print(f"Origin: {example['origin']} | Fault: {example['fault_label']} | Cause label: {example['rca_label']}")
display(pd.DataFrame([
    {"KPI": name, "Type": channel["kind"], "Unit": channel["unit"] or "unknown",
     "Samples": len(channel["values"]), "First 8 values": channel["values"][:8]}
    for name, channel in window["channels"].items()
]))
window_values = pd.DataFrame({name: channel["values"] for name, channel in window["channels"].items()})
window_values.insert(0, "relative_time_seconds", [i / window["sampling_rate_hz"] for i in range(len(window_values))])
window_values.index.name = "sample_index"
display(window_values.head(10))

example_channels = [name for name in ["RX_Bytes", "UL_BLER", "UL_SNR"] if name in window["channels"]]
if not example_channels:
    example_channels = [name for name, channel in window["channels"].items() if channel["kind"] == "numeric"][:3]
if example_channels:
    fig, axes = plt.subplots(len(example_channels), 1, figsize=(11, 2.5 * len(example_channels)), squeeze=False)
    annotation = example_record["anomaly"]
    for ax, name in zip(axes[:, 0], example_channels):
        ax.plot(window_values["relative_time_seconds"], window_values[name], label=name)
        start, end = annotation["start_index"], annotation["end_index"]
        if annotation["exists"] and start is not None and end is not None:
            # Half a sample on each side includes both labelled endpoint samples.
            half_step = 0.5 / window["sampling_rate_hz"]
            ax.axvspan(max(0, start / window["sampling_rate_hz"] - half_step),
                       min(window_values["relative_time_seconds"].iloc[-1], end / window["sampling_rate_hz"] + half_step),
                       color="#e45756", alpha=0.2, label="Source-labelled fault interval")
        ax.set_title(name)
        ax.set_ylabel(window["channels"][name]["unit"] or "Unit unknown")
        ax.legend(loc="best")
    axes[-1, 0].set_xlabel("Seconds since this window started")
    fig.suptitle(f"One window — {example['rca_label']}")
    show_plot("annotated_window")
else:
    print("This window has no numeric KPI channels to plot.")

## Numeric distributions and correlations in the illustrative selection

Channel scales differ, so each distribution uses its own axes. Correlation describes association in selected observations and does not prove a fault mechanism.

In [ ]:
if numeric:
    sample[numeric].hist(bins=25, figsize=(14, 3 * ((len(numeric) + 3) // 4)), layout=(-1, 4))
    show_plot("illustrative_distributions")
    correlations = sample[numeric].corr()
    fig, ax = plt.subplots(figsize=(10, 8))
    plot = ax.imshow(correlations.to_numpy(), vmin=-1, vmax=1, cmap="coolwarm")
    ax.set_xticks(range(len(numeric)), numeric, rotation=90)
    ax.set_yticks(range(len(numeric)), numeric)
    ax.set_title("Illustrative cross-channel correlations")
    fig.colorbar(plot, ax=ax, label="Pearson correlation")
    show_plot("illustrative_correlations")

## Representative KPI traces

Display one selected window per available origin. Axes use relative seconds, and source units remain explicit. Labels identify source annotations; plotted traces are not causal evidence or detection performance.

In [ ]:
chosen_channels = [name for name in ["RX_Bytes", "UL_BLER", "UL_SNR"] if name in numeric] or numeric[:3]
representatives = overview.loc[overview["scenario_id"].isin(selected_ids)].groupby("origin", sort=True).head(1)
if chosen_channels:
    fig, axes = plt.subplots(len(representatives), len(chosen_channels), figsize=(5 * len(chosen_channels), 3 * len(representatives)), squeeze=False)
    for row_index, (_, scenario) in enumerate(representatives.iterrows()):
        trace = sample.loc[sample["scenario_id"].eq(scenario["scenario_id"])].sort_values("sample_index")
        for column_index, name in enumerate(chosen_channels):
            ax = axes[row_index, column_index]
            ax.plot(trace["relative_time_seconds"], trace[name])
            ax.set_title(f"{scenario['origin']}: {name}")
            ax.set_xlabel("Relative seconds")
            ax.set_ylabel(schema["channels"][name]["unit"] or "Unit unknown")
    show_plot("representative_traces")

## Text review and recording relationships

Pending text is retained as reference data but does not enter model inputs. Unresolved relationships remain visible and require issue #22 work before scored train/test experiments. Duplicate observation matches are leads for review, not automatically verified shared recordings.

In [ ]:
display(pd.crosstab(overview["origin"], overview["text_review_status"]))
fig, axes = plt.subplots(1, 2, figsize=(11, 4))
overview["text_review_status"].value_counts().plot.bar(ax=axes[0], title="Human text review status")
overview["relationship_status"].value_counts().plot.bar(ax=axes[1], title="Recording relationship status")
show_plot("text_and_lineage_coverage")
print("Exact observation duplicate matches:", len(conversion.get("duplicates", [])))
print("Approved nonempty text:", overview["has_approved_text"].eq("True").sum())
print("Figure directory:", FIGURES)
print("Conversion is not experiment readiness: grouping, splits and safe inference exports remain in issue #22.")